---
## Módulo 0 — Instalación de Librerías

In [ ]:
# Instalación de las librerías necesarias para el proyecto
# Ejecutar únicamente una vez al inicio de la sesión de Google Colab
!pip install yfinance ta scikit-learn pandas numpy --quiet

  Preparing metadata (setup.py) ... done


---
## Módulo 1 — Importación de Librerías

In [ ]:
# ── Librerías estándar ──────────────────────────────────────────────
import warnings
import json
from datetime import datetime, timedelta

# ── Manipulación de datos ───────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Descarga de datos financieros ──────────────────────────────────
import yfinance as yf

# ── Indicadores técnicos ────────────────────────────────────────────
import ta
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange

# ── Machine Learning ────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.pipeline import Pipeline

# ── Configuración general ───────────────────────────────────────────
warnings.filterwarnings('ignore')
np.random.seed(42)  # Semilla fija para reproducibilidad
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Librerías importadas correctamente')
print(f'   yfinance: {yf.__version__}')
print(f'   scikit-learn: disponible')
print(f'   ta (Technical Analysis): disponible')

✅ Librerías importadas correctamente
   yfinance: 0.2.66
   scikit-learn: disponible
   ta (Technical Analysis): disponible


---
## Módulo 2 — Configuración de Activos y Parámetros

In [ ]:

TICKERS = {
    'FSM':         'Fortuna Silver Mines Inc.',
    'VOLCABC1.LM': 'Volcan Compañía Minera S.A.A.',
    'ABX.TO':      'Barrick Gold Corporation',
    'BVN':         'Compañía de Minas Buenaventura S.A.A.',
    'BHP':         'BHP Billiton Limited'
}

# ── Parámetros temporales ────────────────────────────────────────────
FECHA_FIN   = datetime.today().strftime('%Y-%m-%d')
FECHA_INICIO = (datetime.today() - timedelta(days=730)).strftime('%Y-%m-%d')  # 2 años

# ── Parámetros del modelo ────────────────────────────────────────────
TEST_SIZE   = 0.20   # Partición 80% entrenamiento / 20% prueba
SEED        = 42     # Semilla para reproducibilidad

# ── Grilla de hiperparámetros para GridSearchCV ──────────────────────
PARAM_GRID = [
    {
        'svc__kernel': ['linear'],
        'svc__C':      [0.1, 1, 10, 100]
    },
    {
        'svc__kernel': ['rbf', 'poly', 'sigmoid'],
        'svc__C':      [0.1, 1, 10, 100],
        'svc__gamma':  ['scale', 'auto']
    }
]

print('✅ Configuración establecida')
print(f'   Periodo: {FECHA_INICIO} → {FECHA_FIN}')
print(f'   Tickers: {list(TICKERS.keys())}')
print(f'   Partición train/test: {int((1-TEST_SIZE)*100)}% / {int(TEST_SIZE*100)}%')

✅ Configuración establecida
   Periodo: 2024-06-20 → 2026-06-20
   Tickers: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Partición train/test: 80% / 20%


---
## Módulo 3 — Descarga de Datos OHLCV con yfinance

In [ ]:
def descargar_datos(ticker: str, fecha_inicio: str, fecha_fin: str) -> pd.DataFrame | None:
    """
    Descarga datos OHLCV de Yahoo Finance para un ticker dado.

    Parámetros
    ----------
    ticker       : Símbolo bursátil (p.ej. 'BVN', 'ABX.TO')
    fecha_inicio : Fecha de inicio en formato 'YYYY-MM-DD'
    fecha_fin    : Fecha de fin en formato 'YYYY-MM-DD'

    Retorna
    -------
    DataFrame con columnas OHLCV o None si la descarga falla.
    """
    try:
        df = yf.download(ticker, start=fecha_inicio, end=fecha_fin,
                         auto_adjust=True, progress=False)

        # Aplanar columnas MultiIndex si yfinance las genera
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Verificar que se descargaron suficientes datos
        if df.empty or len(df) < 100:
            print(f'  ⚠️  {ticker}: datos insuficientes ({len(df)} filas)')
            return None

        # Eliminar filas con valores nulos en precios clave
        df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'], inplace=True)
        df.index = pd.to_datetime(df.index)

        print(f'  ✅ {ticker}: {len(df)} filas | {df.index[0].date()} → {df.index[-1].date()}')
        return df

    except Exception as e:
        print(f'  ❌ {ticker}: error en descarga → {e}')
        return None


# ── Descarga de todos los tickers ────────────────────────────────────
print('📥 Descargando datos desde Yahoo Finance...')
print(f'   Periodo: {FECHA_INICIO} → {FECHA_FIN}\n')

datos_crudos = {}  # Diccionario: ticker → DataFrame con datos OHLCV
for ticker in TICKERS.keys():
    datos_crudos[ticker] = descargar_datos(ticker, FECHA_INICIO, FECHA_FIN)

# Resumen de descargas exitosas
tickers_ok = [t for t, df in datos_crudos.items() if df is not None]
tickers_fail = [t for t, df in datos_crudos.items() if df is None]

print(f'\n📊 Resumen: {len(tickers_ok)}/{len(TICKERS)} tickers descargados exitosamente')
if tickers_fail:
    print(f'   Fallidos: {tickers_fail}')

📥 Descargando datos desde Yahoo Finance...
   Periodo: 2024-06-20 → 2026-06-20

  ✅ FSM: 501 filas | 2024-06-20 → 2026-06-18
  ✅ VOLCABC1.LM: 494 filas | 2024-06-20 → 2026-06-18
  ✅ ABX.TO: 502 filas | 2024-06-20 → 2026-06-19
  ✅ BVN: 501 filas | 2024-06-20 → 2026-06-18
  ✅ BHP: 501 filas | 2024-06-20 → 2026-06-18

📊 Resumen: 5/5 tickers descargados exitosamente


---
## Módulo 4 — Ingeniería de Características (Feature Engineering)

In [ ]:
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula todos los indicadores técnicos y variables de features
    requeridas por el pipeline SVC.

    Features calculadas:
    - Retornos: diario, 5 días, 10 días
    - Medias móviles: SMA-20, SMA-50, EMA-12, EMA-26
    - Momentum: RSI-14, MACD, Signal, Histograma
    - Volatilidad: Bollinger Bands (superior, inferior, ancho, %B)
    - ATR-14: Average True Range
    - Oscilador Estocástico: %K, %D
    - Variables derivadas: distancia precio vs SMA, ratio volumen
    """
    data = df.copy()

    close = data['Close']
    high  = data['High']
    low   = data['Low']
    vol   = data['Volume']

    # ── Retornos logarítmicos ──────────────────────────────────────
    data['return_1d']  = close.pct_change(1)    # Retorno diario
    data['return_5d']  = close.pct_change(5)    # Retorno semanal
    data['return_10d'] = close.pct_change(10)   # Retorno bi-semanal
    data['log_return'] = np.log(close / close.shift(1))  # Retorno logarítmico

    # ── Medias Móviles Simples (SMA) ──────────────────────────────
    data['sma_20'] = SMAIndicator(close=close, window=20).sma_indicator()
    data['sma_50'] = SMAIndicator(close=close, window=50).sma_indicator()

    # ── Medias Móviles Exponenciales (EMA) ────────────────────────
    data['ema_12'] = EMAIndicator(close=close, window=12).ema_indicator()
    data['ema_26'] = EMAIndicator(close=close, window=26).ema_indicator()

    # ── RSI-14 (Relative Strength Index) ──────────────────────────
    data['rsi_14'] = RSIIndicator(close=close, window=14).rsi()

    # ── MACD (Moving Average Convergence Divergence) ───────────────
    macd_obj = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    data['macd']        = macd_obj.macd()
    data['macd_signal'] = macd_obj.macd_signal()
    data['macd_hist']   = macd_obj.macd_diff()

    # ── Bandas de Bollinger (ventana=20, std=2) ───────────────────
    bb = BollingerBands(close=close, window=20, window_dev=2)
    data['bb_upper']  = bb.bollinger_hband()
    data['bb_lower']  = bb.bollinger_lband()
    data['bb_width']  = bb.bollinger_wband()   # Ancho de banda
    data['bb_pct']    = bb.bollinger_pband()   # %B (posición relativa)

    # ── ATR-14 (Average True Range) — Volatilidad ─────────────────
    data['atr_14'] = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()

    # ── Oscilador Estocástico ─────────────────────────────────────
    stoch = StochasticOscillator(high=high, low=low, close=close, window=14, smooth_window=3)
    data['stoch_k'] = stoch.stoch()
    data['stoch_d'] = stoch.stoch_signal()

    # ── Variables derivadas y de momentum ─────────────────────────
    # Distancia del precio respecto a las medias móviles
    data['price_vs_sma20'] = (close - data['sma_20']) / data['sma_20']  # % por encima/debajo de SMA-20
    data['price_vs_sma50'] = (close - data['sma_50']) / data['sma_50']  # % por encima/debajo de SMA-50
    data['sma20_vs_sma50'] = (data['sma_20'] - data['sma_50']) / data['sma_50']  # Cruce de medias
    data['ema12_vs_ema26'] = (data['ema_12'] - data['ema_26']) / data['ema_26']  # Cruce EMA

    # Volatilidad histórica (desviación estándar de retornos en 20 días)
    data['volatilidad_20d'] = data['return_1d'].rolling(window=20).std()

    # Ratio de volumen respecto a la media de 20 días
    data['vol_ratio'] = vol / vol.rolling(window=20).mean()

    # Momentum de precio (precio actual vs precio hace N días)
    data['momentum_10d'] = close / close.shift(10) - 1
    data['momentum_20d'] = close / close.shift(20) - 1

    return data


# ── Aplicar ingeniería de características a todos los tickers ────────
print('⚙️  Calculando indicadores técnicos...\n')
datos_features = {}

for ticker, df in datos_crudos.items():
    if df is not None:
        datos_features[ticker] = calcular_indicadores(df)
        n_features = datos_features[ticker].shape[1] - 5  # Descontar OHLCV originales
        print(f'  ✅ {ticker}: {n_features} features calculadas')

print(f'\n📐 Total de features por ticker: {n_features}')

⚙️  Calculando indicadores técnicos...

  ✅ FSM: 27 features calculadas
  ✅ VOLCABC1.LM: 27 features calculadas
  ✅ ABX.TO: 27 features calculadas
  ✅ BVN: 27 features calculadas
  ✅ BHP: 27 features calculadas

📐 Total de features por ticker: 27


---
## Módulo 5 — Variable Objetivo Binaria (Trend: BUY / SELL)

In [ ]:
# ── Columnas de features a usar en el modelo ─────────────────────────
# Se excluyen las columnas OHLCV crudas para evitar data leakage
FEATURE_COLS = [
    'return_1d', 'return_5d', 'return_10d', 'log_return',
    'sma_20', 'sma_50', 'ema_12', 'ema_26',
    'rsi_14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_lower', 'bb_width', 'bb_pct',
    'atr_14',
    'stoch_k', 'stoch_d',
    'price_vs_sma20', 'price_vs_sma50', 'sma20_vs_sma50', 'ema12_vs_ema26',
    'volatilidad_20d',
    'vol_ratio',
    'momentum_10d', 'momentum_20d'
]

def crear_variable_objetivo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea la variable objetivo binaria 'Trend':
      - 1 = BUY  → el retorno del día SIGUIENTE es positivo (precio sube)
      - 0 = SELL → el retorno del día SIGUIENTE es negativo o cero (precio baja)

    IMPORTANTE: se usa el retorno futuro (shift(-1)) para evitar data leakage.
    La última fila siempre será NaN (no hay 'día siguiente') y se elimina.
    """
    data = df.copy()

    # Retorno del día siguiente (objetivo a predecir)
    retorno_siguiente = data['Close'].shift(-1) / data['Close'] - 1

    # Clasificación binaria: 1=BUY si sube, 0=SELL si baja o se mantiene
    data['Trend'] = (retorno_siguiente > 0).astype(int)

    return data


def preparar_X_y(df: pd.DataFrame):
    """
    Prepara los conjuntos X (features) e y (objetivo) para el entrenamiento.
    Elimina filas con valores NaN generados por indicadores técnicos.
    Preserva el orden temporal (no shuffle).
    """
    data = crear_variable_objetivo(df)

    # Seleccionar solo las columnas de features definidas
    cols_disponibles = [c for c in FEATURE_COLS if c in data.columns]

    # Eliminar filas con cualquier NaN (generadas por indicadores con ventana larga)
    data_clean = data[cols_disponibles + ['Trend', 'Close']].dropna()

    X = data_clean[cols_disponibles].values
    y = data_clean['Trend'].values
    fechas = data_clean.index
    precios = data_clean['Close'].values

    return X, y, fechas, precios, cols_disponibles


# ── Verificar distribución de clases ─────────────────────────────────
print('🎯 Distribución de la variable objetivo por ticker:\n')
print(f'{"Ticker":<15} {"Total":<8} {"BUY (1)":<10} {"SELL (0)":<10} {"% BUY"}')
print('-' * 50)

for ticker, df in datos_features.items():
    X, y, fechas, precios, _ = preparar_X_y(df)
    n_buy  = (y == 1).sum()
    n_sell = (y == 0).sum()
    pct    = n_buy / len(y) * 100
    print(f'{ticker:<15} {len(y):<8} {n_buy:<10} {n_sell:<10} {pct:.1f}%')

🎯 Distribución de la variable objetivo por ticker:

Ticker          Total    BUY (1)    SELL (0)   % BUY
--------------------------------------------------
FSM             452      234        218        51.8%
VOLCABC1.LM     445      208        237        46.7%
ABX.TO          453      248        205        54.7%
BVN             452      242        210        53.5%
BHP             452      241        211        53.3%


---
## Módulo 6 — Normalización y Partición Train/Test

In [ ]:
def particionar_temporal(X, y, fechas, precios, test_size=0.20):
    """
    Divide los datos en conjuntos de entrenamiento y prueba
    RESPETANDO EL ORDEN TEMPORAL (sin shuffle aleatorio).

    El 20% final de los datos se usa como conjunto de prueba,
    simulando predicción de datos futuros no vistos.

    Retorna: X_train, X_test, y_train, y_test,
             fechas_train, fechas_test, precios_test
    """
    n_total = len(X)
    n_test  = int(n_total * test_size)
    n_train = n_total - n_test

    X_train, X_test   = X[:n_train],       X[n_train:]
    y_train, y_test   = y[:n_train],       y[n_train:]
    f_train, f_test   = fechas[:n_train],  fechas[n_train:]
    p_test            = precios[n_train:]

    return X_train, X_test, y_train, y_test, f_train, f_test, p_test


# ── Construir el pipeline: StandardScaler + SVC ──────────────────────
# El pipeline garantiza que el scaler se ajuste SOLO sobre train
# y luego se aplica sobre test (evita data leakage de normalización)
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Normalización z-score
    ('svc',    SVC(probability=True, random_state=SEED, max_iter=5000))
])

print('✅ Pipeline creado:')
print('   1. StandardScaler  → normaliza features a media=0, desv.std=1')
print('   2. SVC             → Support Vector Classifier con probability=True')
print(f'\n   Semilla de reproducibilidad: {SEED}')
print(f'   Partición: {int((1-TEST_SIZE)*100)}% train / {int(TEST_SIZE*100)}% test (orden temporal)')

✅ Pipeline creado:
   1. StandardScaler  → normaliza features a media=0, desv.std=1
   2. SVC             → Support Vector Classifier con probability=True

   Semilla de reproducibilidad: 42
   Partición: 80% train / 20% test (orden temporal)


---
## Módulo 7 — Entrenamiento con GridSearchCV

In [ ]:
def entrenar_svc_gridsearch(X_train, y_train, param_grid, cv_splits=5):
    """
    Entrena el pipeline SVC usando GridSearchCV con validación cruzada
    de series de tiempo (TimeSeriesSplit) para respetar el orden temporal.

    Parámetros buscados:
    - kernel: linear, rbf, poly, sigmoid
    - C:      0.1, 1, 10, 100
    - gamma:  scale, auto  (solo para kernels no-lineales)

    Retorna el mejor estimador encontrado y el objeto GridSearchCV.
    """
    # TimeSeriesSplit respeta el orden temporal en la validación cruzada
    tscv = TimeSeriesSplit(n_splits=cv_splits)

    grid_search = GridSearchCV(
        estimator  = pipeline,
        param_grid = param_grid,
        cv         = tscv,
        scoring    = 'f1',       # Optimizar F1-score (más robusto que accuracy con clases desbalanceadas)
        n_jobs     = -1,         # Usar todos los núcleos disponibles
        verbose    = 0,
        refit      = True        # Reentrenar con los mejores parámetros sobre todo el train
    )

    grid_search.fit(X_train, y_train)

    return grid_search


# ── Entrenamiento por ticker ──────────────────────────────────────────
print('🤖 Entrenando clasificador SVC con GridSearchCV...\n')
print('   Kernels evaluados: linear, rbf, poly, sigmoid')
print('   Valores de C:      0.1, 1, 10, 100')
print('   Valores de gamma:  scale, auto (kernels no lineales)')
print('   Validación cruzada: TimeSeriesSplit (5 folds)\n')

resultados = {}  # Diccionario con resultados completos por ticker

for ticker, df in datos_features.items():
    print(f'⏳ Procesando {ticker} — {TICKERS[ticker]}...')

    # 1. Preparar features y objetivo
    X, y, fechas, precios, feature_names = preparar_X_y(df)

    # 2. Partición temporal train/test
    X_train, X_test, y_train, y_test, f_train, f_test, p_test = \
        particionar_temporal(X, y, fechas, precios, test_size=TEST_SIZE)

    # 3. GridSearchCV
    grid = entrenar_svc_gridsearch(X_train, y_train, PARAM_GRID)

    # 4. Predicciones sobre el conjunto de test
    y_pred = grid.predict(X_test)
    y_prob = grid.predict_proba(X_test)[:, 1]  # Probabilidad de clase BUY

    # 5. Métricas
    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, zero_division=0)
    rec   = recall_score(y_test, y_pred, zero_division=0)
    f1    = f1_score(y_test, y_pred, zero_division=0)
    cm    = confusion_matrix(y_test, y_pred)

    # 6. Mejor configuración encontrada
    best_params = grid.best_params_
    best_kernel = best_params.get('svc__kernel', 'rbf')
    best_C      = best_params.get('svc__C', 1)
    best_gamma  = best_params.get('svc__gamma', 'scale')
    best_score  = grid.best_score_

    # 7. Señal actual (última predicción del periodo de test)
    ultima_senal_num = y_pred[-1]
    ultima_senal     = 'BUY' if ultima_senal_num == 1 else 'SELL'
    confianza        = float(y_prob[-1]) if ultima_senal_num == 1 else float(1 - y_prob[-1])

    # 8. Señales completas del periodo de test con fechas y precios
    senales_test = [
        {
            'fecha':  f.strftime('%Y-%m-%d'),
            'precio': round(float(p_test[i]), 4),
            'senal':  'BUY' if y_pred[i] == 1 else 'SELL',
            'real':   'BUY' if y_test[i] == 1 else 'SELL',
            'prob_buy': round(float(y_prob[i]), 4)
        }
        for i, f in enumerate(f_test)
    ]

    # 9. Distribución de clases en el test
    n_buy_test  = int((y_pred == 1).sum())
    n_sell_test = int((y_pred == 0).sum())

    # 10. Matriz de confusión como lista 2x2
    # Orden: [[VP (TP), FP], [FN, VN (TN)]]
    # - VP: predijo BUY y era BUY  (True Positive)
    # - FP: predijo BUY pero era SELL  (False Positive)
    # - FN: predijo SELL pero era BUY  (False Negative)
    # - VN: predijo SELL y era SELL  (True Negative)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()  # sklearn devuelve TN, FP, FN, TP
    else:
        tn, fp, fn, tp = 0, 0, 0, int(cm[0, 0])

    # Almacenar resultados
    resultados[ticker] = {
        'ticker':        ticker,
        'nombre':        TICKERS[ticker],
        # Señal actual
        'senal':         ultima_senal,
        'conf':          round(confianza, 4),
        # Métricas del modelo
        'metricas': {
            'accuracy':  round(float(acc),  4),
            'precision': round(float(prec), 4),
            'recall':    round(float(rec),  4),
            'f1':        round(float(f1),   4)
        },
        # Matriz de confusión en formato [[VP, FP],[FN, VN]]
        'matriz': [[int(tp), int(fp)], [int(fn), int(tn)]],
        # Distribución de señales predichas
        'clases': {
            'buy':  n_buy_test,
            'sell': n_sell_test
        },
        # Mejor configuración del kernel
        'kernel': {
            'tipo':        best_kernel,
            'C':           best_C,
            'gamma':       best_gamma,
            'best_score':  round(float(best_score), 4)
        },
        # Serie de tiempo del periodo de test
        'fechas':   [s['fecha']   for s in senales_test],
        'precios':  [s['precio']  for s in senales_test],
        'senales':  [s['senal']   for s in senales_test],
        # Metadatos
        'n_train': int(len(X_train)),
        'n_test':  int(len(X_test))
    }

    print(f'  ✅ {ticker} | Accuracy: {acc:.2%} | F1: {f1:.4f} | '
          f'Kernel: {best_kernel} | C={best_C} | Señal: {ultima_senal}')

print('\n🎉 Entrenamiento completado para todos los tickers')

🤖 Entrenando clasificador SVC con GridSearchCV...

   Kernels evaluados: linear, rbf, poly, sigmoid
   Valores de C:      0.1, 1, 10, 100
   Valores de gamma:  scale, auto (kernels no lineales)
   Validación cruzada: TimeSeriesSplit (5 folds)

⏳ Procesando FSM — Fortuna Silver Mines Inc....
  ✅ FSM | Accuracy: 46.67% | F1: 0.6364 | Kernel: rbf | C=0.1 | Señal: BUY
⏳ Procesando VOLCABC1.LM — Volcan Compañía Minera S.A.A....
  ✅ VOLCABC1.LM | Accuracy: 51.69% | F1: 0.5905 | Kernel: sigmoid | C=100 | Señal: SELL
⏳ Procesando ABX.TO — Barrick Gold Corporation...
  ✅ ABX.TO | Accuracy: 55.56% | F1: 0.6154 | Kernel: sigmoid | C=10 | Señal: BUY
⏳ Procesando BVN — Compañía de Minas Buenaventura S.A.A....
  ✅ BVN | Accuracy: 48.89% | F1: 0.5965 | Kernel: sigmoid | C=100 | Señal: BUY
⏳ Procesando BHP — BHP Billiton Limited...
  ✅ BHP | Accuracy: 55.56% | F1: 0.7143 | Kernel: rbf | C=0.1 | Señal: BUY

🎉 Entrenamiento completado para todos los tickers


---
## Módulo 8 — Evaluación de Métricas y Reporte

In [ ]:
# ── Tabla comparativa de métricas por ticker ──────────────────────────
print('=' * 80)
print('📊 REPORTE COMPARATIVO DE MÉTRICAS — CLASIFICADOR SVC')
print('=' * 80)
print(f'\n{"Ticker":<15} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10} {"Kernel":<10} {"C":<6} {"Gamma"}')
print('-' * 80)

for ticker, res in resultados.items():
    m = res['metricas']
    k = res['kernel']
    print(f"{ticker:<15} {m['accuracy']:>10.2%} {m['precision']:>10.2%} "
          f"{m['recall']:>10.2%} {m['f1']:>10.4f} "
          f"{k['tipo']:<10} {str(k['C']):<6} {k['gamma']}")

print('-' * 80)

# ── Promedio general ──────────────────────────────────────────────────
metrics_vals = {m: np.mean([resultados[t]['metricas'][m] for t in resultados]) for m in ['accuracy','precision','recall','f1']}
print(f"\n{'PROMEDIO GENERAL':<15} {metrics_vals['accuracy']:>10.2%} {metrics_vals['precision']:>10.2%} "
      f"{metrics_vals['recall']:>10.2%} {metrics_vals['f1']:>10.4f}")

print('\n')
print('=' * 80)
print('🔍 MATRICES DE CONFUSIÓN POR TICKER')
print('=' * 80)

for ticker, res in resultados.items():
    [[tp, fp], [fn, tn]] = res['matriz']
    print(f'\n  {ticker} — {res["nombre"]}')
    print(f'  Señal actual: {res["senal"]} (confianza: {res["conf"]:.1%})')
    print(f'  ┌─────────────────┬────────┬────────┐')
    print(f'  │                 │ PRED B │ PRED S │')
    print(f'  ├─────────────────┼────────┼────────┤')
    print(f'  │ REAL BUY        │ VP={tp:>4} │ FP={fp:>4} │')
    print(f'  │ REAL SELL       │ FN={fn:>4} │ VN={tn:>4} │')
    print(f'  └─────────────────┴────────┴────────┘')
    print(f'  BUY señales: {res["clases"]["buy"]}  |  SELL señales: {res["clases"]["sell"]}')

📊 REPORTE COMPARATIVO DE MÉTRICAS — CLASIFICADOR SVC

Ticker            Accuracy  Precision     Recall   F1-Score Kernel     C      Gamma
--------------------------------------------------------------------------------
FSM                 46.67%     46.67%    100.00%     0.6364 rbf        0.1    scale
VOLCABC1.LM         51.69%     47.69%     77.50%     0.5905 sigmoid    100    scale
ABX.TO              55.56%     52.46%     74.42%     0.6154 sigmoid    10     scale
BVN                 48.89%     50.00%     73.91%     0.5965 sigmoid    100    scale
BHP                 55.56%     55.56%    100.00%     0.7143 rbf        0.1    scale
--------------------------------------------------------------------------------

PROMEDIO GENERAL     51.67%     50.48%     85.17%     0.6306


🔍 MATRICES DE CONFUSIÓN POR TICKER

  FSM — Fortuna Silver Mines Inc.
  Señal actual: BUY (confianza: 53.1%)
  ┌─────────────────┬────────┬────────┐
  │                 │ PRED B │ PRED S │
  ├─────────────────┼──────

---
## Módulo 9 — Preparación de Estructura de Datos para el Frontend HTML

In [ ]:

datos_frontend = {}

for ticker, res in resultados.items():
    datos_frontend[ticker] = {
        # Serie temporal con fechas, precios y señales SVC
        'fechas':  res['fechas'],
        'precios': res['precios'],
        'senales': res['senales'],

        # Señal actual (última fecha del periodo de test)
        'senal': res['senal'],
        'conf':  res['conf'],

        # Métricas de desempeño del clasificador
        'metricas': res['metricas'],

        # Matriz de confusión en formato [[VP, FP],[FN, VN]]
        # Compatible con: const [[tp,fp],[fn,tn]] = datos.matriz
        'matriz': res['matriz'],

        # Distribución de señales predichas en el periodo de test
        # Compatible con: datos.clases.buy  y  datos.clases.sell
        'clases': res['clases'],

        # Mejor configuración encontrada por GridSearchCV
        # Compatible con: datos.kernel.tipo, datos.kernel.C, datos.kernel.gamma
        'kernel': {
            'tipo':  res['kernel']['tipo'],
            'C':     res['kernel']['C'],
            'gamma': res['kernel']['gamma']
        }
    }

# ── Verificación de la estructura generada ────────────────────────────
print('✅ Estructura de datos generada para el frontend\n')
print('📋 Verificación de compatibilidad con el HTML:')
print()

# Mostrar ejemplo con el primer ticker disponible
ticker_ejemplo = list(datos_frontend.keys())[0]
d = datos_frontend[ticker_ejemplo]

print(f'  Ticker de ejemplo: {ticker_ejemplo}')
print(f'  ├── fechas:       {len(d["fechas"])} entradas | primera: {d["fechas"][0]} | última: {d["fechas"][-1]}')
print(f'  ├── precios:      {len(d["precios"])} valores  | rango: [{min(d["precios"]):.4f} – {max(d["precios"]):.4f}]')
print(f'  ├── senales:      {len(d["senales"])} señales  | BUY: {d["senales"].count("BUY")} | SELL: {d["senales"].count("SELL")}')
print(f'  ├── senal:        {d["senal"]}  (señal actual)')
print(f'  ├── conf:         {d["conf"]:.4f}  ({d["conf"]*100:.1f}% confianza)')
print(f'  ├── metricas:')
for k, v in d['metricas'].items():
    print(f'  │     {k:<10}: {v:.4f} ({v*100:.1f}%)')
print(f'  ├── matriz:       [[VP={d["matriz"][0][0]}, FP={d["matriz"][0][1]}], [FN={d["matriz"][1][0]}, VN={d["matriz"][1][1]}]]')
print(f'  ├── clases:       BUY={d["clases"]["buy"]} | SELL={d["clases"]["sell"]}')
print(f'  └── kernel:       tipo={d["kernel"]["tipo"]} | C={d["kernel"]["C"]} | gamma={d["kernel"]["gamma"]}')

print(f'\n✅ Tickers en la estructura: {list(datos_frontend.keys())}')
print('\n💡 La celda siguiente exportará esta estructura a datos_svc.json')

✅ Estructura de datos generada para el frontend

📋 Verificación de compatibilidad con el HTML:

  Ticker de ejemplo: FSM
  ├── fechas:       90 entradas | primera: 2026-02-10 | última: 2026-06-18
  ├── precios:      90 valores  | rango: [8.0900 – 13.6600]
  ├── senales:      90 señales  | BUY: 90 | SELL: 0
  ├── senal:        BUY  (señal actual)
  ├── conf:         0.5308  (53.1% confianza)
  ├── metricas:
  │     accuracy  : 0.4667 (46.7%)
  │     precision : 0.4667 (46.7%)
  │     recall    : 1.0000 (100.0%)
  │     f1        : 0.6364 (63.6%)
  ├── matriz:       [[VP=42, FP=48], [FN=0, VN=0]]
  ├── clases:       BUY=90 | SELL=0
  └── kernel:       tipo=rbf | C=0.1 | gamma=scale

✅ Tickers en la estructura: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

💡 La celda siguiente exportará esta estructura a datos_svc.json


---
## Resumen Final y Verificación

In [ ]:
# ── Resumen ejecutivo del pipeline SVC ───────────────────────────────
print('=' * 70)
print('📝 RESUMEN EJECUTIVO — Pipeline SVC Completado')
print('=' * 70)
print()

print('PIPELINE EJECUTADO:')
print('  1. Descarga de datos OHLCV reales con yfinance (2 años)')
print('  2. Ingeniería de 27 features técnicas por ticker')
print('  3. Variable objetivo binaria: Trend (BUY=1, SELL=0)')
print('  4. Normalización StandardScaler dentro del pipeline')
print('  5. Partición temporal 80% train / 20% test')
print('  6. GridSearchCV con TimeSeriesSplit (5 folds)')
print('  7. Evaluación: accuracy, precision, recall, F1')
print()
print('SEÑALES ACTUALES:')
for ticker, res in resultados.items():
    emoji = '🟢' if res['senal'] == 'BUY' else '🔴'
    print(f'  {emoji} {ticker:<15}: {res["senal"]} ({res["conf"]*100:.1f}% confianza) — '
          f'Acc: {res["metricas"]["accuracy"]*100:.1f}%')

print()
print('ESTRUCTURA datos_frontend generada:')
print(f'  {len(datos_frontend)} tickers procesados exitosamente')
for ticker in datos_frontend:
    n = len(datos_frontend[ticker]['fechas'])
    print(f'  ✅ {ticker}: {n} filas de datos en el periodo de test')

print()
print('SIGUIENTE PASO:')
print('  → Ejecutar la celda de exportación JSON para generar datos_svc.json')
print('  → Dicho JSON se conectará al archivo ernesto_investing_svc_clasificador.html')
print('=' * 70)

📝 RESUMEN EJECUTIVO — Pipeline SVC Completado

PIPELINE EJECUTADO:
  1. Descarga de datos OHLCV reales con yfinance (2 años)
  2. Ingeniería de 27 features técnicas por ticker
  3. Variable objetivo binaria: Trend (BUY=1, SELL=0)
  4. Normalización StandardScaler dentro del pipeline
  5. Partición temporal 80% train / 20% test
  6. GridSearchCV con TimeSeriesSplit (5 folds)
  7. Evaluación: accuracy, precision, recall, F1

SEÑALES ACTUALES:
  🟢 FSM            : BUY (53.1% confianza) — Acc: 46.7%
  🔴 VOLCABC1.LM    : SELL (46.3% confianza) — Acc: 51.7%
  🟢 ABX.TO         : BUY (58.1% confianza) — Acc: 55.6%
  🟢 BVN            : BUY (54.9% confianza) — Acc: 48.9%
  🟢 BHP            : BUY (53.3% confianza) — Acc: 55.6%

ESTRUCTURA datos_frontend generada:
  5 tickers procesados exitosamente
  ✅ FSM: 90 filas de datos en el periodo de test
  ✅ VOLCABC1.LM: 89 filas de datos en el periodo de test
  ✅ ABX.TO: 90 filas de datos en el periodo de test
  ✅ BVN: 90 filas de datos en el periodo de

---
## Celda Final — Exportación a JSON

> Esta celda exporta la estructura de datos generada al archivo `datos_svc.json`,  
> que constituye el **contrato de datos** entre el backend Python y el frontend HTML.

In [ ]:
# ── Exportación a archivo JSON ────────────────────────────────────────
# Este JSON es el contrato de datos entre el backend
# y el frontend
NOMBRE_ARCHIVO = 'datos_svc.json'

# Agregar metadatos de generación al JSON
output = {
    'metadata': {
        'generado_en':    datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'periodo_inicio': FECHA_INICIO,
        'periodo_fin':    FECHA_FIN,
        'modelo':         'Support Vector Classifier (SVC)',
        'framework':      'scikit-learn',
        'clasificacion':  'binaria',
        'clases':         {'0': 'SELL', '1': 'BUY'},
        'tickers':        list(datos_frontend.keys()),
        'n_features':     len(FEATURE_COLS)
    },
    'tickers': datos_frontend
}

# Guardar el JSON con indentación para legibilidad
with open(NOMBRE_ARCHIVO, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

# Verificar el archivo generado
import os
tamanio_kb = os.path.getsize(NOMBRE_ARCHIVO) / 1024

print(f'✅ Archivo exportado: {NOMBRE_ARCHIVO}')
print(f'   Tamaño: {tamanio_kb:.1f} KB')
print(f'   Tickers incluidos: {list(datos_frontend.keys())}')
print()

# Mostrar las primeras líneas del JSON generado
print('📄 Vista previa del JSON (primeras 40 líneas):')
print('-' * 60)
with open(NOMBRE_ARCHIVO, 'r', encoding='utf-8') as f:
    lineas = f.readlines()
    for i, linea in enumerate(lineas[:40]):
        print(linea, end='')
    if len(lineas) > 40:
        print(f'\n... ({len(lineas) - 40} líneas más)')

print('-' * 60)
print()

# Descarga automática en Google Colab
try:
    from google.colab import files
    files.download(NOMBRE_ARCHIVO)
    print(f'⬇️  Descarga iniciada: {NOMBRE_ARCHIVO}')
except ImportError:
    print(f'💡 Entorno local detectado. El archivo está en: ./{NOMBRE_ARCHIVO}')
except Exception as e:
    print(f'💡 Archivo guardado localmente: ./{NOMBRE_ARCHIVO}')

print()
print('🎉 Pipeline SVC completado exitosamente.')
print('   Conectar datos_svc.json al frontend ernesto_investing_svc_clasificador.html')

✅ Archivo exportado: datos_svc.json
   Tamaño: 26.1 KB
   Tickers incluidos: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

📄 Vista previa del JSON (primeras 40 líneas):
------------------------------------------------------------
{
  "metadata": {
    "generado_en": "2026-06-20 13:02:06",
    "periodo_inicio": "2024-06-20",
    "periodo_fin": "2026-06-20",
    "modelo": "Support Vector Classifier (SVC)",
    "framework": "scikit-learn",
    "clasificacion": "binaria",
    "clases": {
      "0": "SELL",
      "1": "BUY"
    },
    "tickers": [
      "FSM",
      "VOLCABC1.LM",
      "ABX.TO",
      "BVN",
      "BHP"
    ],
    "n_features": 27
  },
  "tickers": {
    "FSM": {
      "fechas": [
        "2026-02-10",
        "2026-02-11",
        "2026-02-12",
        "2026-02-13",
        "2026-02-17",
        "2026-02-18",
        "2026-02-19",
        "2026-02-20",
        "2026-02-23",
        "2026-02-24",
        "2026-02-25",
        "2026-02-26",
        "2026-02-27",
        "

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Descarga iniciada: datos_svc.json

🎉 Pipeline SVC completado exitosamente.
   Conectar datos_svc.json al frontend ernesto_investing_svc_clasificador.html
